# Fase 1 - Ingenieria de Datos y EDA (Spark SQL)

Limpieza de campos sucios (altura, peso, valor de transferencia), tratamiento de nulos y analisis de correlacion, usando Spark DataFrame API / Spark SQL.

In [1]:
import os
os.environ.setdefault('JAVA_HOME', '/opt/homebrew/opt/openjdk@17')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName('ScoutingEDA') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

df = spark.read.csv('../data/raw/merged_players.csv', header=True, inferSchema=True)
print((df.count(), len(df.columns)))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/13 19:19:39 WARN Utils: Your hostname, MacBook-de-Rodrigo.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/07/13 19:19:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/13 19:19:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


(91672, 88)


In [2]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- UID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Rec: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Inf: string (nullable = true)
 |-- Club: string (nullable = true)
 |-- Based: string (nullable = true)
 |-- Nat: string (nullable = true)
 |-- Height: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Position: string (nullable = true)
 |-- Transfer Value: string (nullable = true)
 |-- Media Description: string (nullable = true)
 |-- Rc Injury: string (nullable = true)
 |-- Preferred Foot: string (nullable = true)
 |-- Left Foot: string (nullable = true)
 |-- Right Foot: string (nullable = true)
 |-- Caps: integer (nullable = true)
 |-- AT Apps: string (nullable = true)
 |-- AT Gls: string (nullable = true)
 |-- AT Lge Apps: string (nullable = true)
 |-- AT Lge Gls: string (nullable = true)
 |-- Team: string (nullable = true)
 |-- Yth Apps: string 

## Limpieza de campos sucios

- `Height` (`5'9"`) -> centimetros.
- `Weight` (`65 kg`) -> numero.
- `Transfer Value`: es un **rango** con sufijos K/M (`'250K$ - 2.5M$'`) y puede venir como `'Not for Sale'` o valor unico (`'0$'`). Se parsea a limite inferior, superior y un punto estimado (promedio).

In [3]:
# Altura: pies'pulgadas" -> cm
df = df.withColumn(
    'Height_cm',
    F.round(
        (F.regexp_extract('Height', r"(\d+)'", 1).cast('double') * 12 +
         F.regexp_extract('Height', r"'(\d+)\"", 1).cast('double')) * 2.54,
        1
    )
)

# Peso: 'NN kg' -> numero
df = df.withColumn('Weight_kg', F.regexp_extract('Weight', r'(\d+)\s*kg', 1).cast('double'))

df.select('Height', 'Height_cm', 'Weight', 'Weight_kg').show(5)

+------+---------+------+---------+
|Height|Height_cm|Weight|Weight_kg|
+------+---------+------+---------+
| 5'9""|    175.3| 65 kg|     65.0|
| 5'4""|    162.6| 56 kg|     56.0|
| 6'2""|    188.0| 77 kg|     77.0|
| 6'1""|    185.4| 74 kg|     74.0|
| 6'0""|    182.9| 72 kg|     72.0|
+------+---------+------+---------+
only showing top 5 rows


In [4]:
def parse_money_to_number(col_expr):
    """Convierte '250K$' o '2.5M$' o '0$' a double."""
    number = F.regexp_extract(col_expr, r'([\d.]+)', 1).cast('double')
    suffix = F.regexp_extract(col_expr, r'([KM])\$', 1)
    multiplier = F.when(suffix == 'K', 1e3).when(suffix == 'M', 1e6).otherwise(1.0)
    return number * multiplier

is_range = F.col('Transfer Value').contains(' - ')
is_not_for_sale = F.col('Transfer Value') == 'Not for Sale'

low_part = F.when(is_range, F.split('Transfer Value', ' - ').getItem(0)).otherwise(F.col('Transfer Value'))
high_part = F.when(is_range, F.split('Transfer Value', ' - ').getItem(1)).otherwise(F.col('Transfer Value'))

df = df.withColumn('Transfer_Value_Low', F.when(is_not_for_sale, None).otherwise(parse_money_to_number(low_part)))
df = df.withColumn('Transfer_Value_High', F.when(is_not_for_sale, None).otherwise(parse_money_to_number(high_part)))
df = df.withColumn(
    'Transfer_Value_num',
    F.when(is_not_for_sale, None).otherwise((F.col('Transfer_Value_Low') + F.col('Transfer_Value_High')) / 2)
)

df.select('Transfer Value', 'Transfer_Value_Low', 'Transfer_Value_High', 'Transfer_Value_num').show(15, truncate=False)

+--------------+------------------+-------------------+------------------+
|Transfer Value|Transfer_Value_Low|Transfer_Value_High|Transfer_Value_num|
+--------------+------------------+-------------------+------------------+
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0               |0.0                |0.0               |
|0$            |0.0      

## Tratamiento de nulos y atipicos

In [5]:
# 'Rec' (recomendacion del scout) usa '- - -' como marcador de nulo y es categorica, no numerica
df = df.withColumn('Rec', F.when(F.col('Rec').cast('string') == '- - -', None).otherwise(F.col('Rec')))

# Estas si son conteos de carrera/seleccion: '-' como marcador de nulo, numericas.
# inferSchema las detecto como numericas, por lo que comparamos como string antes de castear
# (en modo ANSI, comparar una columna numerica contra '-' directamente lanza un error de cast).
dash_null_cols = ['Caps', 'AT Apps', 'AT Gls', 'AT Lge Apps', 'AT Lge Gls', 'Yth Apps', 'Yth Gls']
for c in dash_null_cols:
    df = df.withColumn(c, F.when(F.col(c).cast('string') == '-', None).otherwise(F.col(c).cast('double')))

null_check_cols = ['Rec'] + dash_null_cols + ['Transfer_Value_num', 'Height_cm', 'Weight_kg']
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in null_check_cols]).show()

+-----+----+-------+------+-----------+----------+--------+-------+------------------+---------+---------+
|  Rec|Caps|AT Apps|AT Gls|AT Lge Apps|AT Lge Gls|Yth Apps|Yth Gls|Transfer_Value_num|Height_cm|Weight_kg|
+-----+----+-------+------+-----------+----------+--------+-------+------------------+---------+---------+
|91672|   0|  25820| 43274|      26102|     43392|   84203|  88770|              2051|        0|        0|
+-----+----+-------+------+-----------+----------+--------+-------+------------------+---------+---------+



In [6]:
# 'Not for Sale' y jugadores sin Transfer Value valido: se documentan y se excluyen del modelado de regresion
n_not_for_sale = df.filter(F.col('Transfer Value') == 'Not for Sale').count()
n_null_value = df.filter(F.col('Transfer_Value_num').isNull()).count()
print(f"Not for Sale: {n_not_for_sale}")
print(f"Transfer_Value_num nulo: {n_null_value}")

Not for Sale: 2051
Transfer_Value_num nulo: 2051


## Analisis de correlacion

Atributos tecnicos/fisicos/mentales (FM attributes) vs. `Transfer_Value_num`.

In [7]:
# 'Nat.1' es un duplicado de nombre de columna (la 2da columna 'Nat' del CSV, atributo de habilidad
# natural). El punto en el nombre confunde a APIs como df.stat.corr (lo interpreta como campo anidado).
df = df.withColumnRenamed('Nat.1', 'Habilidad_Natural')

attribute_cols = [
    'Acc','Wor','Vis','Thr','Tec','Tea','Tck','Str','Sta','TRO','Ref','Pun','Pos','Pen','Pas','Pac',
    '1v1','OtB','Habilidad_Natural','Mar','L Th','Lon','Ldr','Kic','Jum','Hea','Han','Fre','Fla','Fir','Fin','Ecc',
    'Dri','Det','Dec','Cro','Cor','Cnt','Cmp','Com','Cmd','Bra','Bal','Ant','Agi','Agg','Aer','Vers',
    'Temp','Spor','Prof','Pres','Loy','Inj Pr','Imp M','Dirt','Amb','Ada','Cons','Cont',
]

correlations = []
for c in attribute_cols:
    corr = df.stat.corr(c, 'Transfer_Value_num')
    correlations.append((c, corr))

corr_df = spark.createDataFrame(correlations, ['atributo', 'correlacion_con_transfer_value'])
corr_df.orderBy(F.desc('correlacion_con_transfer_value')).show(20, truncate=False)

+--------+------------------------------+
|atributo|correlacion_con_transfer_value|
+--------+------------------------------+
|Cmp     |0.17493662042456046           |
|Ant     |0.1618314467896632            |
|Tea     |0.15370188757270214           |
|Bal     |0.1536126423741744            |
|Vis     |0.15234812215706525           |
|Pas     |0.15003935177788635           |
|Cnt     |0.14704945747684628           |
|Wor     |0.14451514327824333           |
|Amb     |0.14379708570698396           |
|Sta     |0.13958628341105386           |
|Tec     |0.1365863550710163            |
|Pac     |0.13114668865362655           |
|Prof    |0.12749144399078902           |
|Pen     |0.1271741737890464            |
|Str     |0.1247867071530421            |
|Bra     |0.12437050232092321           |
|Fir     |0.1231878379084709            |
|Dri     |0.12115665592388873           |
|Agi     |0.11933515906807513           |
|Acc     |0.11222455697822369           |
+--------+------------------------

## Exportar dataset limpio

In [8]:
df.write.mode('overwrite').parquet('../data/processed/players_clean.parquet')
print('Guardado en data/processed/players_clean.parquet')

26/07/13 19:19:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/07/13 19:19:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , UID, Name, Rec, DOB, Inf, Club, Based, Nat, Height, Weight, Age, Position, Transfer Value, Media Description, Rc Injury, Preferred Foot, Left Foot, Right Foot, Caps, AT Apps, AT Gls, AT Lge Apps, AT Lge Gls, Team, Yth Apps, Yth Gls, Acc, Wor, Vis, Thr, Tec, Tea, Tck, Str, Sta, TRO, Ref, Pun, Pos, Pen, Pas, Pac, 1v1, OtB, Nat.1, Mar, L Th, Lon, Ldr, Kic, Jum, Hea, Han, Fre, Fla, Fir, Fin, Ecc, Dri, Det, Dec, Cro, Cor, Cnt, Cmp, Com, Cmd, Bra, Bal, Ant, Agi, Agg, Aer, Vers, Temp, Spor, Prof, Pres, Loy, Inj Pr, Imp M, Dirt, Amb, Ada, Cons, Cont, Media Handling
 Schema: _c0, UID, Name, Rec, DOB, Inf, Club, Based, Nat, Height, Weight, Age, Position, Transfer Value, Media Description, Rc Injury, Preferred Foot, Left Foot, Right Foot, Caps, AT Apps, AT Gls, AT Lge Apps, AT Lge Gls, Team, Yth Apps, Yth Gls, Acc, Wor, Vis, Thr, Tec, Tea, Tck, Str, Sta, TRO, Ref, Pun, Pos, Pen, Pas, Pac, 1v1, OtB, Nat.

Guardado en data/processed/players_clean.parquet
